In [1]:
# Install dependencies if you haven't already:
# %pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu124
# %pip install ultralytics opencv-python scikit-learn matplotlib numpy
# %pip install tensorflow==2.16.1 tf-keras
# %pip install ml-dtypes==0.3.2

import cv2
import glob
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from ultralytics import YOLO
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from torch.utils.data import TensorDataset, DataLoader

# 1. Configuration
DATA_DIR = "images"
# TEST_VIDEO_PATH = "test_video.mp4" # Update with your test video
# OUTPUT_VIDEO_PATH = "rep_count_output.mp4"
POSE_CONFIDENCE = 0.5

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Load YOLOv8 Nano Pose (downloads automatically if missing)
print("Loading YOLOv8 Pose model...")
yolo_pose_model = YOLO('yolov8n-pose.pt')

Using device: cpu
Loading YOLOv8 Pose model...


In [4]:
def process_image_dataset(data_dir, yolo_model):
    #  Your_Project_Folder/              
    # ├── Rep_Count.ipynb      
    # ├── test_video.mp4                
    # └── data/
    #   └── images/
    #       ├── jump_middle/       
    #       ├── lunges_middle/
    #       ├── pushup_middle/       
    #       ├── pushup_start/  
    #       ├── situp_middle/
    #       ├── situp_start/  
    #       ├── squat_middle/       
    #       └── squat_start/            
    X_data = []
    y_data = []
    classes = ['jump_middle', 'lunges_left', 'pushup_middle', 'pushup_start', 'situp_middle', 'situp_start', 'squat_middle', 'squat_start']
    
    for class_name in classes:
        # Check for multiple common image extensions
        image_paths = glob.glob(f"{data_dir}/{class_name}/*.*")
        print(f"Found {len(image_paths)} images for {class_name}")
        
        for img_path in image_paths:
            frame = cv2.imread(img_path)
            if frame is None: 
                continue
                
            # Run YOLO inference
            results = yolo_model(frame, verbose=False)
            
            if len(results) > 0 and results[0].keypoints is not None:
                kpts_data = results[0].keypoints.data
                if len(kpts_data) > 0:
                    kpts = kpts_data[0].cpu().numpy() # Shape: (17, 3)
                    
                    # Only use high-confidence detections
                    if np.mean(kpts[:, 2]) >= POSE_CONFIDENCE:
                        # Normalize coordinates so the model is resolution-independent
                        h, w = frame.shape[:2]
                        kpts_normalized = kpts.copy()
                        kpts_normalized[:, 0] /= w
                        kpts_normalized[:, 1] /= h
                        
                        X_data.append(kpts_normalized.flatten())
                        y_data.append(class_name)
                        
    return np.array(X_data, dtype=np.float32), np.array(y_data)

# Process the dataset
print("Extracting keypoints from images...")
X_raw, y_raw = process_image_dataset(DATA_DIR, yolo_pose_model)

if len(X_raw) == 0:
    raise ValueError("No valid keypoints found. Check your image paths and dataset folders.")

# Encode labels
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y_raw)
num_classes = len(label_encoder.classes_)

# Split into Train and Validation sets
X_train, X_val, y_train, y_val = train_test_split(X_raw, y_encoded, test_size=0.2, random_state=42)

# Convert to PyTorch Dataloaders
train_dataset = TensorDataset(torch.tensor(X_train), torch.tensor(y_train, dtype=torch.long))
val_dataset = TensorDataset(torch.tensor(X_val), torch.tensor(y_val, dtype=torch.long))

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

print(f"Dataset ready: {len(X_train)} train samples, {len(X_val)} validation samples.")

Extracting keypoints from images...
Found 51 images for jump_middle
Found 26 images for lunges_left
Found 43 images for pushup_middle
Found 51 images for pushup_start
Found 53 images for situp_middle
Found 41 images for situp_start
Found 16 images for squat_middle
Found 61 images for squat_start
Dataset ready: 263 train samples, 66 validation samples.


In [6]:
import os
os.environ["TF_USE_LEGACY_KERAS"] = "1"
import tensorflow as tf
import tf_keras as keras
import numpy as np

print("Building the Model")
keras_model = keras.Sequential([
    keras.layers.Input(batch_shape=(1, 51)),
    keras.layers.Dense(128),
    keras.layers.ReLU(max_value=6.0),   
    keras.layers.Dense(64),
    keras.layers.ReLU(max_value=6.0),   
    keras.layers.Dense(8, activation = "softmax")
])

keras_model.compile(
    optimizer='adam',
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=False),
    metrics=['accuracy']
)

print("Training")
X_train_np = np.array(X_train, dtype=np.float32)
y_train_np = np.array(y_train, dtype=np.int32)
early_stopping = keras.callbacks.EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True)
keras_model.fit(X_train_np, y_train_np, epochs=100, batch_size=32, validation_split=0.1, callbacks=[early_stopping])

print("Quantizing Model")
converter = tf.lite.TFLiteConverter.from_keras_model(keras_model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]

def representative_dataset():
    yield [np.zeros((1, 51), dtype=np.float32)]
    yield [np.ones((1, 51), dtype=np.float32)]
    for i in range(len(X_train_np)):
        yield [X_train_np[i].reshape(1, 51)]

converter.representative_dataset = representative_dataset
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type = tf.int8
converter.inference_output_type = tf.int8

tflite_path = "rep_counter_int8.tflite"
with open(tflite_path, "wb") as f:
    f.write(converter.convert())

print(f"Saved INT8 model to {tflite_path}")

Building the Model
Training
Epoch 1/100
8/8 [==============================] - 0s 11ms/step - loss: 2.0513 - accuracy: 0.1398 - val_loss: 1.9726 - val_accuracy: 0.2593
Epoch 2/100
8/8 [==============================] - 0s 3ms/step - loss: 1.9240 - accuracy: 0.3771 - val_loss: 1.8946 - val_accuracy: 0.4444
Epoch 3/100
8/8 [==============================] - 0s 3ms/step - loss: 1.8345 - accuracy: 0.5297 - val_loss: 1.7915 - val_accuracy: 0.5926
Epoch 4/100
8/8 [==============================] - 0s 3ms/step - loss: 1.7259 - accuracy: 0.5975 - val_loss: 1.6918 - val_accuracy: 0.6296
Epoch 5/100
8/8 [==============================] - 0s 3ms/step - loss: 1.6240 - accuracy: 0.6356 - val_loss: 1.5976 - val_accuracy: 0.6296
Epoch 6/100
8/8 [==============================] - 0s 4ms/step - loss: 1.5203 - accuracy: 0.6398 - val_loss: 1.5084 - val_accuracy: 0.7778
Epoch 7/100
8/8 [==============================] - 0s 3ms/step - loss: 1.3984 - accuracy: 0.6695 - val_loss: 1.4065 - val_accuracy: 0.629

INFO:tensorflow:Assets written to: /tmp/tmpocgthm2e/assets


Saved INT8 model to rep_counter_int8.tflite


/home/eding/aifitnessmirrordemo/.venv/lib/python3.13/site-packages/tensorflow/lite/python/convert.py:846: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(
fully_quantize: 0, inference_type: 6, input_inference_type: INT8, output_inference_type: INT8
